In [ ]:
import sqlite3
import pandas as pd
import numpy as np

from datetime import datetime

In [ ]:
DB_PATH = "field_attendance.db"

conn = sqlite3.connect(
    DB_PATH
)

cursor = conn.cursor()

print("Database Created")

Database Created


In [ ]:
cursor.execute("""

CREATE TABLE IF NOT EXISTS employees(

    employee_id TEXT PRIMARY KEY,

    employee_name TEXT,

    department TEXT,

    embedding_id INTEGER,

    enrollment_date TEXT

)

""")

conn.commit()

In [ ]:
cursor.execute("""

CREATE TABLE IF NOT EXISTS attendance_queue(

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    employee_id TEXT,

    employee_name TEXT,

    timestamp TEXT,

    latitude REAL,

    longitude REAL,

    confidence REAL,

    liveness_passed INTEGER,

    sync_status INTEGER,

    device_id TEXT

)

""")

conn.commit()

In [ ]:
def register_employee(

    employee_id,

    employee_name,

    department,

    embedding_id

):

    cursor.execute(

        """

        INSERT OR REPLACE

        INTO employees

        VALUES(

            ?,?,?,?,?

        )

        """,

        (

            employee_id,

            employee_name,

            department,

            embedding_id,

            datetime.now().isoformat()

        )

    )

    conn.commit()

In [ ]:
register_employee(

    "EMP001",

    "Yoko Ono",

    "Field Operations",

    0

)

register_employee(

    "EMP002",

    "Colin Powell",

    "Field Operations",

    1

)

In [ ]:
employees = pd.read_sql(

    """

    SELECT *

    FROM employees

    """,

    conn

)

employees

,employee_id,employee_name,department,embedding_id,enrollment_date
0,EMP001,Yoko Ono,Field Operations,0,2026-06-04T12:32:55.935520
1,EMP002,Colin Powell,Field Operations,1,2026-06-04T12:32:55.942892


In [ ]:
def create_attendance_event(

    employee_id,

    employee_name,

    confidence,

    latitude,

    longitude,

    liveness_passed,

    device_id

):

    cursor.execute(

        """

        INSERT INTO attendance_queue(

            employee_id,

            employee_name,

            timestamp,

            latitude,

            longitude,

            confidence,

            liveness_passed,

            sync_status,

            device_id

        )

        VALUES(

            ?,?,?,?,?,?,?,?,?

        )

        """,

        (

            employee_id,

            employee_name,

            datetime.now().isoformat(),

            latitude,

            longitude,

            confidence,

            int(liveness_passed),

            0,

            device_id

        )

    )

    conn.commit()

In [ ]:
def already_marked_today(

    employee_id

):

    today = datetime.now().strftime(

        "%Y-%m-%d"

    )

    cursor.execute(

        """

        SELECT COUNT(*)

        FROM attendance_queue

        WHERE

        employee_id=?

        AND

        DATE(timestamp)=?

        """,

        (

            employee_id,

            today

        )

    )

    count = cursor.fetchone()[0]

    return count > 0

In [ ]:
def mark_attendance(

    employee_id,

    employee_name,

    confidence,

    latitude,

    longitude,

    liveness_passed,

    device_id

):

    if already_marked_today(

        employee_id

    ):

        return "Already Marked"

    create_attendance_event(

        employee_id,

        employee_name,

        confidence,

        latitude,

        longitude,

        liveness_passed,

        device_id

    )

    return "Attendance Marked"

In [ ]:
print(

    mark_attendance(

        "EMP001",

        "Yoko Ono",

        0.96,

        17.385,

        78.486,

        True,

        "ANDROID_001"

    )

)

Already Marked


In [ ]:
attendance = pd.read_sql(

    """

    SELECT *

    FROM attendance_queue

    """,

    conn

)

attendance

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id
0,2,EMP001,Jelena Dokic,2026-06-04T12:27:07.205871,17.385,78.486,0.9998,1,0,ANDROID_001


In [ ]:
pending = pd.read_sql(

    """

    SELECT *

    FROM attendance_queue

    WHERE sync_status=0

    """,

    conn

)

pending

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id
0,2,EMP001,Jelena Dokic,2026-06-04T12:27:07.205871,17.385,78.486,0.9998,1,0,ANDROID_001


In [ ]:
def mark_synced(

    record_id

):

    cursor.execute(

        """

        UPDATE attendance_queue

        SET sync_status=1

        WHERE id=?

        """,

        (

            record_id,

        )

    )

    conn.commit()

In [ ]:
mark_synced(1)

In [ ]:
def purge_synced_records():

    cursor.execute(

        """

        DELETE

        FROM attendance_queue

        WHERE sync_status=1

        """

    )

    conn.commit()

In [ ]:
purge_synced_records()

In [ ]:
attendance = pd.read_sql(

    """

    SELECT *

    FROM attendance_queue

    """,

    conn

)

attendance.to_csv(

    "attendance_export.csv",

    index=False

)

print("Exported")

Exported


In [ ]:
authentication_result = {

    "success": True,

    "employee_name": "Jelena Dokic",

    "confidence_score": 0.9998,

    "confidence_level": "HIGH",

    "liveness": True

}

In [ ]:
employee_map = {

    "Jelena Dokic":"EMP001",

    "Yoko Ono":"EMP002",

    "Colin Powell":"EMP003"

}

In [ ]:
if authentication_result["success"]:

    employee_name = authentication_result[
        "employee_name"
    ]

    if employee_name not in employee_map:

        print(
            "Employee Not Registered"
        )

    else:

        employee_id = employee_map[
            employee_name
        ]

        result = mark_attendance(

            employee_id,

            employee_name,

            authentication_result[
                "confidence_score"
            ],

            17.385,

            78.486,

            authentication_result[
                "liveness"
            ],

            "ANDROID_001"

        )

        print(result)

else:

    print(
        "Authentication Failed"
    )

Already Marked


In [ ]:
dashboard = pd.read_sql(

    """

    SELECT

    employee_name,

    timestamp,

    confidence,

    sync_status

    FROM attendance_queue

    """,

    conn

)

dashboard

,employee_name,timestamp,confidence,sync_status
0,Jelena Dokic,2026-06-04T12:27:07.205871,0.9998,0


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil

shutil.copy(

    "field_attendance.db",

    "/content/drive/MyDrive/field_attendance.db"

)

print("Database Saved")

Database Saved


In [ ]:
import sqlite3

conn = sqlite3.connect(
    "field_attendance.db"
)

print("Reconnected")

Reconnected


In [ ]:
import pandas as pd

pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

,name
0,employees
1,attendance_queue
2,sqlite_sequence


In [ ]:
attendance = pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    """,

    conn

)

attendance

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id
0,2,EMP001,Jelena Dokic,2026-06-04T12:27:07.205871,17.385,78.486,0.9998,1,0,ANDROID_001


In [ ]:
attendance = pd.read_sql(
    """
    SELECT *
    FROM attendance_queue
    """,
    conn
)

print(attendance.shape)

attendance.head()

(1, 10)


,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id
0,2,EMP001,Jelena Dokic,2026-06-04T12:27:07.205871,17.385,78.486,0.9998,1,0,ANDROID_001


In [ ]:
attendance = pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    """,

    conn

)

attendance.to_csv(

    "/content/drive/MyDrive/attendance_export.csv",

    index=False

)

print("CSV Saved")

CSV Saved


In [ ]:
conn.commit()
conn.close()